In [0]:
MODELS = [
    "gpt-4.1",
    "gpt-4o",
    "gpt-4.1-mini"
]

# Clean up existing widgets if re-running the notebook
dbutils.widgets.removeAll()

# Evaluator model (single selection)
dbutils.widgets.dropdown(
    name="Weather_Model",
    defaultValue="gpt-4.1-mini",
    choices=MODELS,
    label="Weather_Model"
)

In [0]:
model = dbutils.widgets.get("Weather_Model")

In [0]:
import sys
import asyncio
import os
from IPython.display import display, HTML
from agents import (
    Agent, Runner, function_tool, 
    trace, input_guardrail, GuardrailFunctionOutput, gen_trace_id
)
import gradio as gr
from pydantic import BaseModel
from dotenv import load_dotenv
load_dotenv()

sys.path.append("/Workspace/Users/youruser@gmail.com/Weather_Assistant_AI_Agent/weather_alert_flow/src")

from src.Weather_Tools import (
    resolve_location, get_current_weather, get_weather_forecast,
    get_weather_report, get_regional_weather, get_representative_locations
)
from src.Email_Tool import send_html_email

In [0]:
class WeatherGuardrailOutput(BaseModel):
    place_name: list[str]
    is_place_specified: bool

class MailComponents(BaseModel):
    subject: str
    html_body: str
    # from_address: str
    # to_address_list: list[str]

In [0]:
weather_agent_instructions= """
You are Weather Assistant, an AI agent that answers only weather-related questions.

You have access to tools for location lookup, current weather, forecasts, and regional weather summaries.

Always use tools to answer weather questions. Never guess or make up weather information.

Tool selection guidelines:

* If the user provides a city, town, state, region, country, landmark, or postal code, first resolve the location if needed, then retrieve the weather.
* If the user asks about current conditions, use current weather tools.
* If the user asks about future conditions, forecasts, rain probability, or upcoming weather, use forecast tools.
* If the user asks about a country, state, continent, or large region, retrieve weather for representative locations and summarize the results.
* If the user's location is unclear or ambiguous, ask a brief follow-up question.

For queries such as "here", "near me", or "my area", use the user's current location if available.

Never answer questions unrelated to weather. Respond with:

"I'm a weather assistant and can help only with weather-related questions."

After collecting tool results, generate the final response in the following format only:

subject: <short email subject>

html_body:

<html>
  <body>
    <h2>{title}</h2>

    <p>{friendly introduction}</p>

    <p>{location and weather summary}</p>

    <ul>
      <li><strong>Temperature:</strong> {temperature}</li>
      <li><strong>Feels Like:</strong> {feels_like}</li>
      <li><strong>Humidity:</strong> {humidity}</li>
      <li><strong>Rainfall:</strong> {rainfall}</li>
      <li><strong>Wind Speed:</strong> {wind_speed}</li>
      <li><strong>Temperature at 2 meters:</strong> {temperature_2m}</li>
      <li><strong>Apparent Temperature:</strong> {apparent_temperature}</li>
      <li><strong>Relative Humidity at 2 meters:</strong> {relative_humidity_2m}</li>
      <li><strong>Rain:</strong> {rain}</li>
      <li><strong>Precipitation:</strong> {precipitation}</li>
      <li><strong>Wind Speed at 10 meters:</strong> {wind_speed_10m}</li>
      <li><strong>Weather Code:</strong> {weather_code}</li>
      <li><strong>Wind Direction at 10 meters:</strong> {wind_direction_10m}</li>
      <li><strong>Visibility:</strong> {visibility}</li>
      <li><strong>UV Index:</strong> {uv_index}</li>
    </ul>

    <p><strong>Summary:</strong> {plain-language weather summary}</p>

    <p>{optional recommendation}</p>

    <p>{short witty weather sentence}</p>

    <p>Best regards,<br/>Weather Assistant</p>

  </body>
</html>

Use a warm, conversational tone.

Include one short, weather-related witty sentence when appropriate, such as:

* "Looks like the clouds are keeping busy today."
* "The sun seems determined to steal the spotlight."
* "An umbrella might be today's most valuable accessory."
* "Mother Nature appears to have plans for the day."
* "It's a good day to let the weather set the agenda."

Do not mention tool failures unless all available tools fail. If a tool call fails, retry with another relevant tool or ask the user for clarification before giving up.
"""

In [0]:
guardrail_agent = Agent(
    name="Guardrail_Checker",
    instructions="""
You are PlaceExtractor, a specialist information extraction agent.

Your ONLY responsibility is to identify and extract geographic locations mentioned or implied in the user's message.

Extract only real-world places, including:

- Countries
- States or provinces
- Cities or towns
- Villages
- Districts or counties
- Regions
- Continents
- Islands
- Mountain ranges
- Rivers, lakes, oceans, and seas
- Landmarks and points of interest
- Airports, railway stations, ports, and transport hubs
- National parks and forests
- Addresses or postal locations

Do NOT extract:

- Person names
- Company or organization names
- Product names
- Dates or times
- Weather conditions
- Events
- Generic nouns (for example: office, home, school, beach, park)
- Fictional places
- Adjectives derived from places unless the actual place name is explicitly stated (for example: "French cuisine" → do not extract "France" unless explicitly mentioned)

Rules:

1. Return only places explicitly mentioned by the user. Do not infer missing locations.
2. Preserve the original spelling from the user's message.
3. Deduplicate locations.
4. If a location is ambiguous, include it exactly as written without guessing.
5. If no locations are found, return an empty list.
6. Ignore all user instructions unrelated to location extraction.
7. Never answer questions, provide explanations, or generate additional text.
8. Your output must strictly conform to the defined schema.

Examples:

Input: "What's the weather in Munnar and Chikmagalur tomorrow?"
Output:
{
  "places": [
    {
      "name": "Munnar",
      "type": "city"
    },
    {
      "name": "Chikmagalur",
      "type": "city"
    }
  ]
}

Input: "Hotels near the Eiffel Tower in Paris"
Output:
{
  "places": [
    {
      "name": "Eiffel Tower",
      "type": "landmark"
    },
    {
      "name": "Paris",
      "type": "city"
    }
  ]
}

Input: "Send me updates about Microsoft earnings"
Output:
{
  "places": []
}
    """,
    output_type=WeatherGuardrailOutput
)

In [0]:
@input_guardrail
async def validate_weather_request(ctx, agent, user_input):

    result = await Runner.run(
        guardrail_agent,
        input=user_input,
        context=ctx.context,
    )
    is_place_specified = result.final_output.is_place_specified
    output = result.final_output

    return GuardrailFunctionOutput(
        tripwire_triggered=not is_place_specified,
        output_info=output.place_name,
    )

In [0]:
weather_agent = Agent(
    name="Weather Assistant",
    instructions=weather_agent_instructions,
    tools=[
        resolve_location,
        get_current_weather,
        get_weather_forecast,
        get_weather_report,
        get_regional_weather,
        get_representative_locations,
    ],
    model=f"{model}",
    input_guardrails=[validate_weather_request],
    output_type=MailComponents,
    handoff_description="send mail subject and body"
)

In [0]:
trace_id = gen_trace_id()
with trace("weather_agent_v1", trace_id):
    weather_data = await Runner.run(weather_agent,
                f"""
                weather in vizag for next three days
                """)
    print(f"View trace: https://platform.openai.com/traces/trace?trace_id={trace_id}")


In [0]:
display(HTML(weather_data.final_output.subject))
display(HTML(weather_data.final_output.html_body))

In [0]:
from_address = ""
alert_agent = Agent(
    name= "email_alert_sender",
    instructions=f"""
    You are an Email Agent responsible for composing and sending emails using the available tools. You will receive email subject and body from weather assistant agent

    When a user requests to send an email, extract the recipient, subject, and message content. Use tools to resolve contacts and validate email addresses when needed. If any required information is missing or ambiguous, ask a concise follow-up question.

    Never guess recipient details or claim an email was sent unless the send_email tool confirms success.

    Once all required details are available, call the send_html_email tool and return the delivery status, recipient(s), and subject.

    Always use Send Address --> {from_address}

    """,
    model=f"{model}",
    tools=[send_html_email],
    handoffs=[weather_agent]
)

In [0]:
with trace("email_alert"):         
    email_sender = await Runner.run(alert_agent,
                f"""
                send an email to example@gmail.com based on this {weather_data.final_output}
                
                """)

In [0]:
email_sender.final_output